# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это первый шаг. Здесь агент должен научиться заходить в иглу = прорыв.

Похоже влажные мечты о том, что 18-ый агент будет по скорости и памяти таким же, как и 17-ый, канули в лету. Придётся признать, что просто даже из-за увеличения токенов с 4 до 36 будет медленее и больше памяти жраться. Пока верным способом снизить потребление памяти и повысить скрость - это перейти на переиспользование `VisionHead`, `RenderHead` из `WorldModel`.

В данной серии пока переиспользования нет, хочется посмотреть, что будет когда `prediction_target='next_obs'` и `action_plan_lv_encoding='separate'`.

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 5
    generation_ind = 0
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    ####
    
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.agent.parent = None
    HP.agent.sequence_length = 4 # length observation chain agent incepts
    HP.agent.ob_shape = (1, 178, 152) 
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.vision_head = dict(grid=(6, 6), features_counts=(16, 32, 64, 128))
    HP.agent.action_plan_lv_encoding = 'separate'
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.attention_backend = 'EFFICIENT_ATTENTION'
    HP.agent.prediction_target = 'next_obs'
    HP.agent.render_heads_count = 4
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_preprocessed_obs = False
    HP.video.capture_env_rams = None
    HP.video.capture_env_ram_patches = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['last_life', 'full_igloo', 'bailey_right_at_the_igloo_door', 'temperature_10'],
        ['last_life', 'full_igloo', 'bailey_very_near_igloo_door', 'temperature_10'],  
        ['last_life', 'full_igloo', 'bailey_near_center', 'temperature_10'],
        ['last_life', 'one_remaining_igloo', 'bailey_near_center', 'temperature_10'],
        ['last_life', 'three_remaining_igloo', 'bailey_near_center', 'temperature_20'],
        ['last_life', 'half_igloo', 'bailey_near_center'],
        ['last_life', 'half_igloo'],
        ['last_life'],
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

# Results
<TBD>

В такой конфигурации явно пободрее агенты стали. Стало ощутимо больше прорывов по сравнению с `18n_study_02.1` и `18n_study_01`. Правда до результатов `17e_study_23.1` всё равно ещё неблизко. Там были уверенные взятия 4k очков многими агентами, здесь только один умудрился. А также несколько агентов уверенно закрепились на прохождении 2-х уровней. Здесь же только один раз эту планку взял, но потом растерялся. Единственный момент - там было 33 запуска, а здесь всего лишь 16...

Возможно, я зря надеюсь увидеть результаты аналогичные `17e_study_23.1` с пол-пинка. Всё таки тут модель сложнее получается, и возможно ей требуется больше времени на обучения. И типа 6 млн шагов недостаточно. Может такое быть? Вполне. Ещё один довод в копилку того, что надо переписпользовать `VisionHead`, `RenderHead`.

<img src="./img/levels_passed.png">
<img src="./img/reward.png">
<img src="./img/episode_r.png">

**Выводы**
1) надо поппробовать ещё вариант, как было в 17-ом агенте - там action_embs имели causal маску.
2) надо грести в сторону переиспользования `VisionHead`, `RenderHead`.